In [2]:
import math
from random import shuffle
from copy import deepcopy
import torch
import torch.nn.functional as tfunc
import matplotlib.pyplot as plt

torch_device = "cpu"
if torch.cuda.is_available():
    print("CUDA is available")
    torch_device = "cuda"
elif torch.backends.mps.is_available():
    print("MPS is available")
    torch_device = "mps"

torch.set_default_device(torch_device)

MPS is available


# Building Dataset

In [ ]:
UID_TO_UNAME = {
    "1211781489931452447": "Wordle",
    "329382986405642240": "evandabest_",
    "1416259149997670433": "joeisaloserandnotgettingoo_34014",
    "518633088180420609": "kingwooziee",
    "834249758159798273": "mango2875",
    "603328024263262208": "oof123",
    "179745514080829440": "plazmama",
    "458421510596460554": "rbtro",
    "511655545321685019": "vhazey",
    "723005048040325140": "whorsey.",
    "601380778684841994": "yannaner",
}
sentences = open("../messages.txt", "r").read().splitlines()
# Remove links(for now, will add later for more advanced models)
sentences = [sentence for sentence in sentences if not sentence.startswith("https://")]
chars = sorted(list(set("".join(sentences))))
char_to_indx = {char: indx+2 for indx, char in enumerate(chars)}
char_to_indx["<S>"] = 0
char_to_indx["<E>"] = 1
indx_to_char = {indx: char for char, indx in char_to_indx.items()}

def train_dev_test_split(words, train_percentage, dev_percentage): # Test percentage is implied since this is a strict 3 way split
    shuffled_words = deepcopy(words)
    shuffle(shuffled_words)
    train_dev_split_point = math.ceil(len(words) * train_percentage)
    dev_val_split_point = math.floor(
        train_dev_split_point + (len(words) * dev_percentage)
    )
    return (
        shuffled_words[:train_dev_split_point],
        shuffled_words[train_dev_split_point:dev_val_split_point],
        shuffled_words[dev_val_split_point:],
    )

def create_dataset(sentences, char_to_indx, block_size = 3):
    # Replace all user mentions @{user.id} i.e. @8212341892348 with their username @{username} i.e. @noscoperkillstreak
    for idx in range(len(sentences)):
        for user_id, username in UID_TO_UNAME.items():
            sentences[idx] = sentences[idx].replace(user_id, username)

    # first_run = True
    inputs, labels = [], []
    for sentence in sentences:
        context = [0] * block_size
        for label in tuple(sentence) + ("<E>",):
            label_indx = char_to_indx[label]
            labels.append(label_indx)
            # if first_run:
            #     print("".join(indx_to_char[indx] for indx in context), "------>", label)

            context = context[1:] + [label_indx]
            inputs.append(context)

        # if first_run: print("="*(block_size*2))
        # first_run = False

    return torch.tensor(inputs), torch.tensor(labels)

BLOCK_SIZE = 128
train_words, dev_words, test_words = train_dev_test_split(sentences, 0.8, 0.1)
train_inputs, train_labels = create_dataset(train_words, char_to_indx, block_size=BLOCK_SIZE)
dev_inputs, dev_labels = create_dataset(dev_words, char_to_indx, block_size=BLOCK_SIZE)
test_inputs, test_labels = create_dataset(test_words, char_to_indx, block_size=BLOCK_SIZE)
print(train_inputs.shape)

torch.Size([301310, 128])


# Initializing Model Params

In [ ]:
# Params
gen = torch.Generator(device=torch_device).manual_seed(2147483647)
CHARS_NUM = len(chars)
EMBED_NUM = 200
HIDDEN_NUM = 1024
DOWN_SCALE_PARAMS = (0.5, 0.1, 0.01)

# Uniform configurable params
C = torch.randn((CHARS_NUM, EMBED_NUM), requires_grad=True) # Lookup table
weights1 = (torch.randn((BLOCK_SIZE * EMBED_NUM, HIDDEN_NUM), requires_grad=True) * DOWN_SCALE_PARAMS[0]).detach().requires_grad_()
weights2 = (torch.randn((HIDDEN_NUM, CHARS_NUM), requires_grad=True) * DOWN_SCALE_PARAMS[1]).detach().requires_grad_()
bias2 = (torch.randn(CHARS_NUM, requires_grad=True) * DOWN_SCALE_PARAMS[2]).detach().requires_grad_()

batch_norm_gain = torch.ones((1, HIDDEN_NUM), requires_grad=True)
batch_norm_bias = torch.zeros((1, HIDDEN_NUM), requires_grad=True)
batch_norm_running_mean = torch.zeros((1, HIDDEN_NUM))
batch_norm_running_std = torch.ones((1, HIDDEN_NUM))

parameters = [C, weights1, weights2, bias2, batch_norm_gain, batch_norm_bias]
print([len(param) for param in parameters])
print(f"TOTAL PARAMS: {sum([param.nelement() for param in parameters])}")

NameError: name 'torch' is not defined

# Training Model

In [ ]:
EPOCHS = 100000
MINI_BATCH_SIZE = 100
LOG_EVERY = 10000

losses = []
learning_rate = 0.1
for indx in range(0, EPOCHS):
    mini_batch_indxs = torch.randint(0, train_inputs.shape[0], (MINI_BATCH_SIZE,))
    mini_batch_inputs = train_inputs[mini_batch_indxs]
    mini_batch_labels = train_labels[mini_batch_indxs]

    embed = C[mini_batch_inputs]
    joined_embed = embed.view(embed.shape[0], embed.shape[1] * embed.shape[2])
    preact = joined_embed @ weights1
    current_mean, current_std = preact.mean(0, keepdim=True), preact.std(0, keepdim=True)
    batch_norm_preact = batch_norm_gain * ((preact - current_mean) / current_std) + batch_norm_bias
    layer1_out = torch.tanh(batch_norm_preact)
    logits = layer1_out @ weights2 + bias2
    loss = tfunc.cross_entropy(logits, mini_batch_labels)

    with torch.no_grad():
        batch_norm_running_mean = 0.99 * batch_norm_running_mean + 0.01 * current_mean
        batch_norm_running_std = 0.99 * batch_norm_running_std + 0.01 * current_std

    for param in parameters:
        param.grad = None

    if indx < 2 * EPOCHS // 3: learning_rate = 0.01 
    loss.backward(retain_graph=True)
    for param in parameters:
        param.data += -learning_rate * param.grad
    
    if indx % LOG_EVERY == 0: print(f"{indx}/{EPOCHS} : {loss.item()}")
    losses.append(loss.log10().item()
)

# Training loss curve plot

In [ ]:
plt.plot(list(range(EPOCHS)), losses)

# Eval Model with Dev Data

In [ ]:
@torch.no_grad()
def calc_loss(inputs, labels):
    embed = C[inputs]
    joined_embed = embed.view(embed.shape[0], embed.shape[1] * embed.shape[2])
    preact = joined_embed @ weights1
    batch_norm_preact = batch_norm_gain * ((preact - batch_norm_running_mean) / batch_norm_running_std) + batch_norm_bias
    layer1_out = torch.tanh(batch_norm_preact)
    logits = layer1_out @ weights2 + bias2
    loss = tfunc.cross_entropy(logits, labels)
    return loss

print(f"Train loss: {calc_loss(train_inputs, train_labels).item()}")
print(f"Dev loss: {calc_loss(dev_inputs, dev_labels).item()}")

In [ ]:
# Best train loss: 1.8334664106369019
# Best dev loss: 2.0098631381988525

# Sample from the model

In [ ]:
NUM_SAMPLES = 20
g = torch.Generator(device=torch_device).manual_seed(2147483647 + 10)
for _ in range(NUM_SAMPLES): 
    out = []
    context = [0] * BLOCK_SIZE
    while True:
        embed = C[torch.tensor([context])]
        joined_embed = embed.view(embed.shape[0], embed.shape[1] * embed.shape[2])
        preact = joined_embed @ weights1
        batch_norm_preact = batch_norm_gain * ((preact - batch_norm_running_mean) / batch_norm_running_std) + batch_norm_bias
        layer1_out = torch.tanh(batch_norm_preact)
        logits = layer1_out @ weights2 + bias2
        probs = tfunc.softmax(logits, dim=1)
        ix = torch.multinomial(probs, num_samples=1, generator=g).item()
        context = context[1:] + [ix]
        out.append(ix)
        if ix == 0:
            break
  
    print(''.join(indx_to_char[i] for i in out)) # decode and print the generated word